# Vehicle Type Classification — Decision Tree

Runs end to end and writes `submissions/submission_decision_tree.csv`.

A single tree: high variance, so depth and leaf size are tuned to prune it. Expect it to be clearly weaker than the other models.

## Setup and preprocessing
* Drop `vehicle_id` (identifier only).
* Drop the 2,500 exact duplicate training rows: they would leak across CV folds and inflate scores (labels never conflict, so nothing is lost).
* No imputation (no missing values), no outlier removal (bounded values), no resampling (balanced classes).
* No scaling needed: tree splits are invariant to monotone feature transforms.

In [1]:
import os, json, numpy as np, pandas as pd, warnings
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold, GridSearchCV, RandomizedSearchCV, cross_val_score, cross_val_predict
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, f1_score
warnings.filterwarnings("ignore")
os.makedirs("submissions", exist_ok=True)
SEED = 42; np.random.seed(SEED)
for d in ["data", ".", "..", "/kaggle/input/vehicle-type-classification"]:
    if os.path.exists(f"{d}/vehicle_train_10000.csv"): DATA_DIR = d; break
else: raise FileNotFoundError("Put the competition CSVs next to this notebook or in ./data")
train  = pd.read_csv(f"{DATA_DIR}/vehicle_train_10000.csv")
test   = pd.read_csv(f"{DATA_DIR}/vehicle_test.csv")
sample = pd.read_csv(f"{DATA_DIR}/vehicle_sample_submission.csv")
TARGET, ID = "vehicle_class", "vehicle_id"
FEATURES = [c for c in test.columns if c != ID]
df = train.drop_duplicates(subset=FEATURES).reset_index(drop=True)   # remove 2,500 exact duplicate rows
X, y, X_test = df[FEATURES], df[TARGET].values, test[FEATURES]
CV5 = StratifiedKFold(5, shuffle=True, random_state=SEED)
print("training rows after de-dup:", len(df), "| class counts:", np.bincount(y))

training rows after de-dup: 7500 | class counts: [2539 2490 2471]


## Hyper-parameter tuning (5-fold stratified CV, accuracy)

In [2]:
grid = {"criterion": ["gini", "entropy"], "max_depth": [3, 4, 5, 6, 7, 8, 10, 12, None],
        "min_samples_leaf": [1, 2, 5, 10, 20, 40], "ccp_alpha": [0, 0.0005, 0.001, 0.002, 0.005]}
search = GridSearchCV(DecisionTreeClassifier(random_state=SEED), grid, cv=CV5, scoring="accuracy", n_jobs=2).fit(X, y)
r = pd.DataFrame(search.cv_results_).sort_values("rank_test_score")
print(r[[c for c in r.columns if c.startswith("param_")] + ["mean_test_score", "std_test_score"]].head(8).round(4).to_string(index=False))

 param_ccp_alpha param_criterion param_max_depth  param_min_samples_leaf  mean_test_score  std_test_score
          0.0010         entropy               7                       5           0.8641          0.0041
          0.0020         entropy               7                      10           0.8640          0.0054
          0.0020         entropy               8                      10           0.8639          0.0057
          0.0020         entropy               7                       5           0.8639          0.0040
          0.0005         entropy               7                       5           0.8639          0.0045
          0.0000            gini               5                       5           0.8636          0.0065
          0.0020         entropy               8                       5           0.8636          0.0041
          0.0010            gini               5                      10           0.8636          0.0048


### Bias–variance illustration: an unpruned tree memorises the training data

In [3]:
full = DecisionTreeClassifier(random_state=SEED)
print("unpruned tree  train acc:", round(full.fit(X, y).score(X, y), 4), "| CV acc:", round(cross_val_score(full, X, y, cv=CV5).mean(), 4))

unpruned tree  train acc: 1.0 | CV acc: 0.8241


## Stability check, overfitting check, and submission

In [4]:
best = search.best_estimator_
print("best params:", search.best_params_, "| 5-fold CV acc:", round(search.best_score_, 4))
rep = cross_val_score(best, X, y, cv=RepeatedStratifiedKFold(n_splits=5, n_repeats=6, random_state=SEED + 1), scoring="accuracy", n_jobs=1)
oof = cross_val_predict(best, X, y, cv=CV5)
print(f"Repeated CV accuracy: {rep.mean():.4f} ± {rep.std():.4f} | OOF macro-F1: {f1_score(y, oof, average='macro'):.4f}")
best.fit(X, y)
print("train accuracy:", round(accuracy_score(y, best.predict(X)), 4), "vs CV", round(search.best_score_, 4))
pred = best.predict(X_test).astype(int)
sub = pd.DataFrame({ID: test[ID], TARGET: pred})
assert list(sub.columns) == list(sample.columns) and len(sub) == len(sample) and (sub[ID].values == sample[ID].values).all() and set(sub[TARGET]) <= {0, 1, 2}
sub.to_csv("submissions/submission_decision_tree.csv", index=False)
json.dump(dict(model="Decision Tree", cv_acc=float(search.best_score_), repeated_cv=float(rep.mean()), repeated_cv_std=float(rep.std()),
               macro_f1=float(f1_score(y, oof, average="macro")), train_acc=float(accuracy_score(y, best.predict(X))), params=str(search.best_params_)), open("submissions/submission_decision_tree_result.json", "w"))
print("written submission_decision_tree.csv | predicted class counts:", np.bincount(pred, minlength=3).tolist())

best params: {'ccp_alpha': 0.001, 'criterion': 'entropy', 'max_depth': 7, 'min_samples_leaf': 5} | 5-fold CV acc: 0.8641


Repeated CV accuracy: 0.8606 ± 0.0090 | OOF macro-F1: 0.8648
train accuracy: 0.8895 vs CV 0.8641
written submission_decision_tree.csv | predicted class counts: [788, 870, 842]
